## 1. Imports & Dataset Loading

In [17]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVR, LinearSVR

df = pd.read_csv('../dataset/predict_prices_dataset.csv')
df.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,Make
0,A1,2017,99,Manual,15735,Petrol,150,55.4,1.4,audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,audi


Same car dataset as the linear, polynomial and GBDT notebooks. Two `sklearn.svm` classes are relevant:
- **`LinearSVR`** — SVR restricted to a linear model, solved by `liblinear`. Scales O(n × d), fast even on tens of thousands of rows.
- **`SVR`** — the general form, solved by `libsvm`. Supports kernels (`linear`, `rbf`, `poly`, `sigmoid`) and scales O(n² – n³), so it's the one to be careful with on larger datasets.

`StandardScaler` is imported too — mandatory for SVR because kernels operate on distances between points, and unscaled features would dominate.

## 2. LinearSVR — Linear Baseline on the Full Dataset

In [18]:
x = df.drop(columns=['price'])
y = df['price']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)
x_train = x_train.copy()
x_test  = x_test.copy()

model_means = y_train.groupby(x_train['model']).mean()
x_train['model_encoded'] = x_train['model'].map(model_means)
x_test['model_encoded']  = x_test['model'].map(model_means)
x_test['model_encoded']  = x_test['model_encoded'].fillna(model_means.mean())

x_train = x_train.drop(columns=['model'])
x_test  = x_test.drop(columns=['model'])

x_train = pd.get_dummies(x_train, columns=['transmission', 'fuelType', 'Make'], drop_first=True)
x_test  = pd.get_dummies(x_test,  columns=['transmission', 'fuelType', 'Make'], drop_first=True)
x_test  = x_test.reindex(columns=x_train.columns, fill_value=0)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled  = scaler.transform(x_test)

model = LinearSVR(
    epsilon=0.0,
    C=1000.0,
    loss='squared_epsilon_insensitive',
    max_iter=20000,
    random_state=42,
)
model.fit(x_train_scaled, y_train)

y_pred_train = model.predict(x_train_scaled)
y_pred_test  = model.predict(x_test_scaled)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       3362.7503   3482.1416
MAE        2224.4602   2221.6331
R²            0.8684      0.8632


Preprocessing choices (all applied *after* the train/test split so the scaler/encoder never see test data):
- **Target encoding for `model`** (~200 unique levels) — mean price per model, computed on train only, mapped to both sets; test-only models fall back to the global mean. One new column instead of ~200 one-hot columns.
- **One-hot for `transmission`, `fuelType`, `Make`** (a handful of levels each) — ~15 extra columns, manageable.
- **`StandardScaler`** on the full feature matrix — required for SVR; fit on train only.

Hyperparameter choices for `LinearSVR`:
- **`epsilon=0.0`** — no dead-zone in the loss; every error contributes. (Default is 0.0 for `LinearSVR`, but 0.1 for `SVR` — inconsistent across the two classes.)
- **`C=1000.0`** — needs to be large because the target ranges up to ~60000. With default `C=1.0`, the regularization term `‖w‖²` dominates the loss and the model underfits severely (R² ≈ 0.42 in that setup); `C=1000` puts the two terms on the same order of magnitude.
- **`loss='squared_epsilon_insensitive'`** — L2 penalty on tube-exit points instead of the default L1. Behaves closer to standard linear regression and is a better fit for RMSE / R² optimization.

## Result Comparison

| Model | Test R² | Test RMSE | Test MAE |
|:------|:-------:|:---------:|:--------:|
| Linear regression (numeric only, polynomial degree 1) | 0.71 | 5067 | 3327 |
| Polynomial regression, degree 3 (numeric only) | 0.81 | 4056 | 2640 |
| **LinearSVR (all features, C=1000, squared loss)** | **0.8632** | **3482** | **2222** |
| Polynomial regression, degree 2 (all features) | 0.918 | 2704 | 1654 |

LinearSVR lands right between polynomial degree 3 (numeric only) and polynomial degree 2 (all features) — the expected ceiling for a strictly linear model with all the features encoded. Any further gain has to come from a non-linear model.

## 3. SVR with RBF Kernel — Adding Non-linearity

In [19]:
model = SVR(
    kernel='rbf',
    C=1000.0,
    epsilon=100.0,
    gamma='scale',
)
model.fit(x_train_scaled, y_train)

y_pred_train = model.predict(x_train_scaled)
y_pred_test  = model.predict(x_test_scaled)

rmse_train = root_mean_squared_error(y_train, y_pred_train)
mae_train  = mean_absolute_error(y_train, y_pred_train)
r2_train   = r2_score(y_train, y_pred_train)

rmse_test = root_mean_squared_error(y_test, y_pred_test)
mae_test  = mean_absolute_error(y_test, y_pred_test)
r2_test   = r2_score(y_test, y_pred_test)

print(f"{'':<8}{'Train':>12}{'Test':>12}")
print(f"{'RMSE':<8}{rmse_train:>12.4f}{rmse_test:>12.4f}")
print(f"{'MAE':<8}{mae_train:>12.4f}{mae_test:>12.4f}")
print(f"{'R²':<8}{r2_train:>12.4f}{r2_test:>12.4f}")

               Train        Test
RMSE       2697.2670   2754.7795
MAE        1448.4234   1464.2437
R²            0.9153      0.9144


Same preprocessed data as the LinearSVR cell above (`x_train_scaled`, `x_test_scaled`, `y_train`, `y_test` are reused). Only the model class changes.

Hyperparameter choices for `SVR`:
- **`kernel='rbf'`** — Gaussian kernel; the standard non-linear default. Maps the data implicitly into an infinite-dimensional space where relationships that are curved in the original space become linear.
- **`C=1000.0`** — same reasoning as LinearSVR (target scale).
- **`epsilon=100.0`** — tube half-width in target units (price). 100 is ~0.2% of the target max — tight enough to keep the fit precise, wide enough to keep the number of support vectors reasonable.
- **`gamma='scale'`** — kernel bandwidth = 1 / (n_features × X.var()). sklearn's default; a sensible starting point that adapts to feature count.

Training on the full ~57k rows took ~8 min — the expected cost of the O(n²) kernel matrix.

## Linear vs RBF

| Setup | Train R² | Test R² | Test RMSE | Test MAE | Gap R² |
|:------|:--------:|:-------:|:---------:|:--------:|:------:|
| LinearSVR (linear model) | 0.8684 | 0.8632 | 3482 | 2222 | 0.005 |
| **SVR + RBF (non-linear model)** | **0.9153** | **0.9144** | **2755** | **1464** | **0.001** |

Big jump: **+5.1 pp on Test R², -21% on Test RMSE, -34% on Test MAE**. And the gap is razor-thin (0.001) — the RBF model generalizes essentially perfectly at this level of complexity.

The +5 pp between the linear and RBF versions is a direct measurement of **how much non-linearity the data actually contains**. Same features, same train/test split, same everything except the model class. The delta is pure non-linearity.

## Full Comparison Across All Four Approaches

| Model | Best Test R² | Best Test RMSE | Best Test MAE | Notes |
|:------|:------------:|:--------------:|:-------------:|:------|
| Polynomial + Ridge (best) | 0.943 | 2253 | 1458 | Degree 3 + Ridge alpha=1000 + FE + outliers |
| **SVR + RBF (this notebook, no tuning)** | **0.914** | **2755** | **1464** | Baseline hyperparameters; ~8 min single fit |
| LightGBM (final, tuned + FE + outliers) | 0.967 | 1708 | 1057 | RandomizedSearchCV + FE + outliers |
| XGBoost (final, tuned + FE + outliers)  | 0.967 | 1709 | 1053 | RandomizedSearchCV + FE + outliers |